# Fit the selected SPARC sample

Fit NFW and Burkert halos to the rotation curves prepared in notebook 1.
Save the fit summaries, radial predictions, individual starts, and configuration for later notebooks.

The summary retains invalid galaxies and failed optimizations.

Li, P., Lelli, F., McGaugh, S., & Schombert, J. (2020). A Comprehensive Catalog of Dark Matter Halo Models for SPARC Galaxies. *The Astrophysical Journal Supplement Series*, 247(1), 31. [DOI: 10.3847/1538-4365/ab700e](https://doi.org/10.3847/1538-4365/ab700e).

See [Section 2.3, “MCMC Simulations,” page 4](https://astroweb.case.edu/ssm/papers/ApJSv247n31.pdf#page=4), in the paragraph beginning “As for halo parameters.” It gives 10 < V200 < 500 km/s and 0 < C200 < 1000. The fit converts speed to mass using M200 = V200³ / (10 G H0). At H0 = 70 km/s/Mpc this gives log10(M200/Msun) of approximately 8.52–13.62. The bounded optimizer includes the speed endpoints and uses a concentration range of 0.001–1000, with 0.001 as the numerical floor for the paper’s open lower limit. These limits are applied to the project’s chi-squared fits.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "models" / "fitting.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
from IPython.display import display
from models.results import FitConfig, result_tables
from models.fitting import fit_sample
INPUT_PATH = PROJECT_ROOT / "data/processed/selected_galaxy_rotation_curves.csv"
FIT_DIR = PROJECT_ROOT / "data/processed/halo_fits"
FIGURE_DIR = PROJECT_ROOT / "figures/halo_fits"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import hashlib
import json
config = FitConfig(g=4.30091e-6, h0=0.07, y_disk=0.5, y_bulge=0.7,
    factors=(0.5, 1., 2.), method="L-BFGS-B",
    optimizer_options={}, agreement_delta_chi2=0.01, bound_tolerance=1e-5)
joined = pd.read_csv(INPUT_PATH)
# These describe the upstream data-processing selection, not additional fit cuts.
SAMPLE_CUTS = {"excluded_quality_flags": [3], "inclination_min": 30, "inclination_max": 80}
assert (joined.quality_flag != 3).all()
assert (joined.inclination - joined.inclination_error >= SAMPLE_CUTS["inclination_min"]).all()
assert (joined.inclination + joined.inclination_error <= SAMPLE_CUTS["inclination_max"]).all()
print(f"Fitting {joined.galaxy.nunique()} galaxies with two models")

Fitting 108 galaxies with two models


In [3]:
results = fit_sample(joined, models=("nfw", "burkert"), config=config)
summary, attempts, predictions = result_tables(results)
assert len(summary) == 2 * joined.galaxy.nunique()
assert not summary.duplicated(["galaxy", "model"]).any()
display(summary.groupby(["model", "status"]).size())

model    status 
burkert  success    108
nfw      success    108
dtype: int64

In [4]:
FIT_DIR.mkdir(parents=True, exist_ok=True)
for filename, table in {"fit_summary.csv": summary, "fit_attempts.csv": attempts,
                        "fit_predictions.csv": predictions}.items():
    table.to_csv(FIT_DIR / filename, index=False)
metadata = {"fit": config.to_dict(), "models": ["nfw", "burkert"],
    "initialization": "maximum observed velocity; same heuristic for both halos; starts clipped to bounds",
    "sample_cuts": SAMPLE_CUTS, "input_path": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "input_sha256": hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()}
(FIT_DIR / "fit_config.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(f"Saved {len(summary)} summary rows, {len(attempts)} attempts, and {len(predictions)} predictions")

Saved 216 summary rows, 1944 attempts, and 4626 predictions
